In [11]:
train_set = '../data/raw/sample_23.csv' # do not touch
test_set = '../data/raw/sample_24.csv' # do not touch

predictions_weight = {
    # '../data/predictions/sklearn_forecast_2024_cluster_specific_combined_case5_clusters.csv': 0.9,
    # '../data/predictions/flo/pred_2024_cluster_xgb(6).csv': 0.1,
    'outputs/agp_forecast_case5_clusters_cluster_specific_combined.csv': 0.35,
    'outputs/pred_2024_cluster_xgb(13).csv': 0.65,
    # '../data/predictions/flo_static_shapelets/pred_2024_cluster_xgb(7).csv': 0.05,
}

In [12]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_absolute_error


def _resolve_path(path_like):
    path = Path(path_like)
    if path.is_absolute():
        return path
    return (Path.cwd() / path).resolve()


def _pick_id_col(dfs):
    preferred = ["ID", "id", "Id", "series_id", "item_id"]
    common_cols = set(dfs[0].columns)
    for df in dfs[1:]:
        common_cols &= set(df.columns)
    for col in preferred:
        if col in common_cols:
            return col
    for col in dfs[0].columns:
        if col in common_cols:
            return col
    raise ValueError("No shared identifier column found across the prediction files.")


def _pick_value_cols(dfs, id_col):
    common_cols = [c for c in dfs[0].columns if c != id_col]
    for df in dfs[1:]:
        common_cols = [c for c in common_cols if c in df.columns and c != id_col]
    if not common_cols:
        raise ValueError("No shared value columns found across the prediction files.")
    return common_cols


prediction_items = list(predictions_weight.items())
if not prediction_items:
    raise ValueError("predictions_weight must contain at least one prediction file.")

prediction_frames = []
for path, weight in prediction_items:
    df_pred = pd.read_csv(_resolve_path(path))
    prediction_frames.append((path, weight, df_pred))

dfs_only = [df for _, _, df in prediction_frames]
id_col = _pick_id_col(dfs_only)
value_cols = _pick_value_cols(dfs_only, id_col)

common_ids = set(dfs_only[0][id_col].dropna())
for df in dfs_only[1:]:
    common_ids &= set(df[id_col].dropna())
if not common_ids:
    raise ValueError("No overlapping IDs found across the prediction files.")
common_ids = sorted(common_ids)

weighted_sum = None
total_weight = 0.0

for path, weight, df_pred in prediction_frames:
    frame = df_pred[df_pred[id_col].isin(common_ids)].copy()
    frame = frame[[id_col] + value_cols].set_index(id_col)
    frame = frame.apply(pd.to_numeric, errors="coerce")
    frame = frame.loc[common_ids]
    if weighted_sum is None:
        weighted_sum = frame * weight
    else:
        weighted_sum = weighted_sum.add(frame * weight, fill_value=0.0)
    total_weight += weight

if total_weight == 0:
    raise ValueError("Total ensemble weight is zero.")

blended_values = weighted_sum / total_weight
blended = blended_values.reset_index()

output_path = _resolve_path("../data/predictions/weighted_average_predictions.csv")
blended.to_csv(output_path, index=False)

test_path = _resolve_path(test_set)
df_true = pd.read_csv(test_path)
if id_col not in df_true.columns:
    raise ValueError(f"Label file does not contain the identifier column '{id_col}'.")

label_value_cols = [c for c in value_cols if c in df_true.columns]
if not label_value_cols:
    raise ValueError("No overlapping value columns found between labels and blended predictions.")

merged = df_true[[id_col] + label_value_cols].merge(
    blended[[id_col] + label_value_cols],
    on=id_col,
    how="inner",
    suffixes=("_true", "_pred"),
)
if merged.empty:
    raise ValueError("No overlapping IDs between the labels and blended predictions.")

y_true = merged[[f"{c}_true" for c in label_value_cols]].to_numpy().ravel()
y_pred = merged[[f"{c}_pred" for c in label_value_cols]].to_numpy().ravel()
mae = mean_absolute_error(y_true, y_pred)

print(f"Saved: {output_path}")
print(f"MAE: {mae:.6f}")
display(blended.head())

Saved: /home/gopes/Documents/Clustering-And-Forecasting-TimeSeries-PlayingGround/data/predictions/weighted_average_predictions.csv
MAE: 3.253904


,ID,2024-01-01,2024-01-02,2024-01-03,2024-01-04,2024-01-05,2024-01-06,2024-01-07,2024-01-08,2024-01-09,...,2024-12-22,2024-12-23,2024-12-24,2024-12-25,2024-12-26,2024-12-27,2024-12-28,2024-12-29,2024-12-30,2024-12-31
0,22,10.953871,10.974406,11.007419,11.019104,10.359841,11.011079,11.671913,11.537084,11.644528,...,12.371853,11.787208,11.784775,11.830725,11.788855,11.557230,11.829197,12.336961,11.815803,11.792907
1,42,42.582770,40.296791,39.474553,41.197352,41.017386,42.902735,45.759552,45.322642,45.280208,...,33.444599,32.118502,32.222913,32.499164,32.656861,33.024883,33.518723,34.013488,33.151614,33.574976
2,56,7.129884,7.680051,7.856326,7.926288,8.396917,9.381425,9.245527,9.542414,9.457652,...,9.289697,8.715408,8.676598,8.895017,8.799355,8.755441,9.078754,9.240562,8.821810,8.842365
3,58,13.445799,13.126960,12.674306,11.511364,12.359513,13.199975,13.256460,12.483818,12.625345,...,18.055613,17.042693,17.016057,17.141043,17.216359,17.326289,17.581842,18.027700,17.305276,17.408574
4,64,2.393749,2.405651,2.410222,2.361631,2.385393,2.451394,2.400265,2.304720,2.326974,...,2.488879,2.512505,2.514563,2.517770,2.507593,2.503782,2.516739,2.489577,2.525706,2.511661
